# Text Preprocessing for NLP

In NLP, preprocessing is the first step to clean and prepare raw text data 
for analysis and machine learning models.

This notebook covers:
1. Text Cleaning
2. Tokenization
3. Normalization (lowercasing, stemming, lemmatization)
4. Stopword Removal
5. Handling contractions & spelling
6. Rare/OOV words
7. Vectorization

## 1. Text Cleaning:

Raw text often contains unwanted characters like punctuation, numbers, HTML tags, URLs, or emojis.  
We clean the text to keep only meaningful words.

- Remove HTML tags
- Remove URLs
- Remove numbers
- Remove punctuation & special characters
- Remove extra whitespace


In [8]:
import re
from bs4 import BeautifulSoup

text = "Hello!!! <br> Visit https://example.com 😊 123"
print("original text:", text)

# Remove HTML tags
clean_text = BeautifulSoup(text, "html.parser").get_text()
print("after removing HTML tags:", clean_text)

# Remove URLs
clean_text = re.sub(r"http\S+", "", clean_text)
print("after removing URLs:", clean_text)

# Remove numbers & punctuation
clean_text = re.sub(r"[^a-zA-Z\s]", "", clean_text)
print("after removing numbers & punctuation:", clean_text)

# removing extra spaces
clean_text = re.sub(r"\s+", " ", clean_text).strip()
print("after removing extra spaces:", clean_text)


original text: Hello!!! <br> Visit https://example.com 😊 123
after removing HTML tags: Hello!!!  Visit https://example.com 😊 123
after removing URLs: Hello!!!  Visit  😊 123
after removing numbers & punctuation: Hello  Visit   
after removing extra spaces: Hello Visit


## 2. Tokenization:
Tokenization splits text into smaller units:
- Word Tokenization → "I love NLP" → ["I", "love", "NLP"]
- Sentence Tokenization → "I love NLP. It's amazing!" → ["I love NLP.", "It's amazing!"]

Why?  
- Models work with tokens, not raw strings.

In [21]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize, sent_tokenize

text = "I love NLP. It's amazing!"
print("Word tokens:", word_tokenize(text))
print("Sentence tokens:", sent_tokenize(text))


Word tokens: ['I', 'love', 'NLP', '.', 'It', "'s", 'amazing', '!']
Sentence tokens: ['I love NLP.', "It's amazing!"]


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 3. Lowercasing:

Lowercasing helps maintain uniformity and reduces duplicates:  
- "Python" and "python" → same token.  
- Sometimes case matters (e.g., "US" vs "us").


In [3]:
text = "NLP is FUN and Powerful!"
print("Lowercased:", text.lower())

Lowercased: nlp is fun and powerful!


## 4. Stopword Removal
- Stopwords = common words that carry little meaning (is, the, an, in).  
- They are often removed to reduce noise in text.

- Example:  
"This is an example" → "example"


In [25]:
from nltk.corpus import stopwords
nltk.download("stopwords")

words = word_tokenize("This is an example sentence for NLP preprocessing")
stop_words = set(stopwords.words("english"))

filtered = [w for w in words if w.lower() not in stop_words]
print("After stopword removal:", filtered)


After stopword removal: ['example', 'sentence', 'NLP', 'preprocessing']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 5. Stemming and Lemmatization
### Stemming
- Cuts words to root form (may not be valid words).
- Example: "studies" → "studi"

### Lemmatization
- Converts words to their dictionary base form using POS tags.
- Example: "studies" → "study"

- >Lemmatization is more accurate, but slower.


In [26]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download("wordnet")
nltk.download("omw-1.4")

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["running", "runs", "ran", "better", "studies"]

print("Stemming:")
for w in words:
    print(w, "->", stemmer.stem(w))

print("\nLemmatization:")
for w in words:
    print(w, "->", lemmatizer.lemmatize(w, pos="v"))

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...


Stemming:
running -> run
runs -> run
ran -> ran
better -> better
studies -> studi

Lemmatization:
running -> run
runs -> run
ran -> run
better -> better
studies -> study


## 6. Handling Contractions & Negations

Contractions expand shortened words:  
- "can't" → "can not"  
- "it's" → "it is"

This helps preserve meaning, especially for sentiment analysis.


In [ ]:
pip install contractions

Note: you may need to restart the kernel to use updated packages.


In [27]:
import contractions

text = "I can't believe it's already done!"
expanded = contractions.fix(text)
print("Before:", text)
print("After:", expanded)


Before: I can't believe it's already done!
After: I cannot believe it is already done!


## 7. Spelling Correction

Spelling mistakes introduce noise. Correcting them improves token consistency.  
Example: "recieve" → "receive"

In [28]:
pip install textblob

Note: you may need to restart the kernel to use updated packages.


In [39]:
from textblob import TextBlob

text = "I havve az speeling errrror"
corrected = str(TextBlob(text).correct())

print("Before:", text)
print("After:", corrected)


Before: I havve az speeling errrror
After: I have a spelling error


## 8. Handling Rare Words / OOV

Rare words or Out-of-Vocabulary (OOV) terms can cause issues in models.  
Approaches:
- Replace with "UNK" token
- Use subword tokenization (Byte Pair Encoding, WordPiece)

In [43]:
# Example 1: Replace rare words with 'UNK' token
from collections import Counter

corpus = ["I love NLP", "NLP is amazing", "Deep learning for NLP"]
words = " ".join(corpus).split()
word_counts = Counter(words)

# Define rare as words that appear only once
rare_words = {w for w, c in word_counts.items() if c == 1}

replaced = [w if w not in rare_words else "UNK" for w in words]
print("Original words:", words)
print("After replacing rare words with 'UNK':", replaced)

# Example 2: Subword tokenization with SentencePiece
# Install sentencepiece if not already installed
# pip install sentencepiece
import sentencepiece as spm

# Write a small corpus to a file for training
with open("sp_corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(line + "\n")

# Train a SentencePiece model (BPE)
spm.SentencePieceTrainer.Train(input='sp_corpus.txt', model_prefix='m', vocab_size=30, model_type='bpe')

# Load the model and encode a sentence
sp = spm.SentencePieceProcessor(model_file='m.model')
example = "NLP is powerful and amazing"
print("Subword tokens:", sp.encode(example, out_type=str))



Original words: ['I', 'love', 'NLP', 'NLP', 'is', 'amazing', 'Deep', 'learning', 'for', 'NLP']
After replacing rare words with 'UNK': ['UNK', 'UNK', 'NLP', 'NLP', 'UNK', 'UNK', 'UNK', 'UNK', 'UNK', 'NLP']
Subword tokens: ['▁NLP', '▁', 'i', 's', '▁', 'p', 'o', 'w', 'e', 'r', 'f', 'u', 'l', '▁', 'a', 'n', 'd', '▁', 'a', 'm', 'a', 'z', 'ing']


## 9. Vectorization

Models need numbers, not text.  
Vectorization converts tokens into numerical representations:

1. Bag of Words (BoW)
2. TF-IDF (reduces weight of frequent words)
3. Word Embeddings (Word2Vec, GloVe, FastText)
4. Contextual Embeddings (BERT, GPT)

In [46]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

docs = ["I love NLP", "NLP loves Python", "Python is great for NLP"]

# Bag of Words
bow = CountVectorizer()
bow_matrix = bow.fit_transform(docs).toarray()
bow_df = pd.DataFrame(bow_matrix, columns=bow.get_feature_names_out())
print("BoW:\n", bow_df)

# TF-IDF
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(docs).toarray()
tfidf_df = pd.DataFrame(tfidf_matrix, columns=tfidf.get_feature_names_out())
print("\nTF-IDF:\n", tfidf_df)

# Word Embeddings with Gensim
from gensim.models import Word2Vec
sentences = [doc.split() for doc in docs]
model = Word2Vec(sentences, vector_size=50, window=2, min_count=1, workers=4)
print("\nWord2Vec embedding for 'NLP':\n", model.wv['NLP'])





BoW:
    for  great  is  love  loves  nlp  python
0    0      0   0     1      0    1       0
1    0      0   0     0      1    1       1
2    1      1   1     0      0    1       1

TF-IDF:
         for     great        is      love     loves       nlp    python
0  0.000000  0.000000  0.000000  0.861037  0.000000  0.508542  0.000000
1  0.000000  0.000000  0.000000  0.000000  0.720333  0.425441  0.547832
2  0.504611  0.504611  0.504611  0.000000  0.000000  0.298032  0.383770

Word2Vec embedding for 'NLP':
 [-1.0724545e-03  4.7286271e-04  1.0206699e-02  1.8018546e-02
 -1.8605899e-02 -1.4233618e-02  1.2917745e-02  1.7945977e-02
 -1.0030856e-02 -7.5267432e-03  1.4761009e-02 -3.0669428e-03
 -9.0732267e-03  1.3108104e-02 -9.7203208e-03 -3.6320353e-03
  5.7531595e-03  1.9837476e-03 -1.6570430e-02 -1.8897636e-02
  1.4623532e-02  1.0140524e-02  1.3515387e-02  1.5257311e-03
  1.2701781e-02 -6.8107317e-03 -1.8928028e-03  1.1537147e-02
 -1.5043275e-02 -7.8722071e-03 -1.5023164e-02 -1.8600845e-03


## 10. Word Embeddings

Embeddings capture semantic meaning of words.  
Example:  
"king" - "man" + "woman" ≈ "queen"

In [47]:
# Example: Word Embeddings - "king" - "man" + "woman" ≈ "queen"
from gensim.models import Word2Vec

# Example sentences for training
sentences = [
    ["king", "queen", "man", "woman", "prince", "princess", "boy", "girl"],
    ["man", "woman", "king", "queen"],
    ["boy", "girl", "prince", "princess"],
    ["king", "man"],
    ["queen", "woman"],
    ["prince", "boy"],
    ["princess", "girl"]
]

# Train a Word2Vec model
model = Word2Vec(sentences, vector_size=50, window=2, min_count=1, workers=2, seed=42)

# Perform the analogy: king - man + woman ≈ queen
result = model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print('king - man + woman ≈', result)

# Show the most similar word
print('Most similar word:', result[0][0])


king - man + woman ≈ [('princess', 0.27577096223831177), ('prince', 0.042404741048812866), ('girl', -0.04414226859807968)]
Most similar word: princess
